# Generalized Random Forest (GRF)

**Generalized Random Forest（GRF）** はRandom Forestを局所的な重み付けの推定器として一般化したもの。

- 与えられた観測点 $x$ に対し、その周囲の観測データを「重み付き」平均で統計量を推定する構造
- 目的変数は一般のM-推定問題に拡張可能（平均だけでなく、分位点、最適政策など）

## 文献

:::{card} 公式

- [[1610.01271] Generalized Random Forests](https://arxiv.org/abs/1610.01271)
- [An introduction to `grf` • grf](https://grf-labs.github.io/grf/articles/grf_guide.html)

:::

:::{card} [一般化ランダムフォレストの理論と統計的因果推論への応用 - Speaker Deck](https://speakerdeck.com/tomoshige_n/ban-hua-randamuhuoresutonoli-lun-totong-ji-de-yin-guo-tui-lun-henoying-yong)

- 非常によくまとまっている資料
- Random Forest / Tree は、サンプルをある基準で重み付けるカーネルを学習するもの、という一般化

:::

## 理論

### Model

outcome $Y_i$が次のような関係になっていると仮定する

$$
Y_i=\tau(X_i) W_i+f(X_i)+\varepsilon_i,
\quad \mathrm{E}\left[\varepsilon_i \mid X_i, W_i\right]=0
$$

- $X_i$ ：共変量
- $W_i \in \{0,1\}$ :処置の有無を示す変数
- $\tau(X_i)$ ：CATE （$\mathrm{E}\left[Y_i(1)-Y_i(0) \mid X_i=x\right]$）

### 通常のRandom Forest

Breiman の random forestは条件付き平均 $\mu(x)=\mathrm{E}\left[Y_i \mid X_i=x\right]$ を推定する

1. **訓練フェーズ** ：共変量と分割点を貪欲法的にサブグループの平均の二乗差 $n_L n_R\left(\bar{y}_L-\bar{y}_R\right)^2$ を最大化するように選び、$B$ 個の木を作る。
2. **予測フェーズ** ：木を平均する。すなわち、目的のサンプル $x$ について、同じ葉ノード $L_b(x)$ におけるoutcomes $Y_i$ の平均を$B$個の木にわたって平均する

$$
\begin{aligned}
\hat{\mu}(x)
& =\sum_{i-1}^n \frac{1}{B} \sum_{b-1}^B Y_i \frac{1\left\{X_i \in L_b(x)\right\}}{\left|L_b(x)\right|} \\
& =\sum_{i-1}^n Y_i \alpha_i(x)
\end{aligned}
$$

すなわち、Random Forestは 重み$\alpha_i(x)$によるoutcomeの荷重和を返す。

### Causal forests

Causal forestsはRobinsonのアプローチ（部分線形モデルの残差回帰）とRandom Forestを組み合わせたもの。

1. **訓練フェーズ：** 推定された部分集団ごとの処置効果の差の二乗、$n_L n_R (\hat{\tau}_L - \hat{\tau}_R)^2$ を最大化するように分割を選択する。ここで、$\hat{\tau}$ は残差同士の回帰によって計算される。
2. **予測フェーズ：** 得られたフォレストの重み $\alpha_i(x)$ を用いて、次のように推定する。

$$
\tau(x) := \operatorname{lm}\left(
Y_i - \hat{m}^{(-i)}(X_i)
\sim
W_i - \hat{e}^{(-i)}(X_i),
\text{ weights } = \alpha_i(x)
\right)
$$

## GRFは欠陥あり？

Atlantic causal inference conferenceの因果推論コンペでの成績は悪かったらしい

（[Ken McAlinnさんはTwitterを使っています: 「因果推定コンテストで回帰分析にボロ負けした話もちゃんと載せてほしい。」 / Twitter](https://twitter.com/kenmcalinn/status/1630768792912506881)）